In [16]:
import numpy as np
from sympy import *

import torch
from torch import nn
# import torch.nn.functional as F
import torch.optim as optim
# import torch.nn.init as init

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

In [17]:
# t, delta, omega = symbols("t delta omega")
t = Symbol("t", positive=True)

m = 1
mu = 5
k = 500

delta = mu/(2*m)
w0 = sqrt(k)
w = sqrt(w0**2 - delta**2)

phi = tan(-delta/w)**(-1)
A = 1/(2 * cos(phi))

display(A)

x = exp(-delta*t)*(2*A*cos(phi + w*t))
display(x.simplify())

-0.595470930409999

-1.19094186082*exp(-2.5*t)*cos(22.220486043289*t - 8.85065980104099)

In [35]:
n = 10

times_train = (np.random.rand(n) * 0.5).astype("float32")
D_train = (np.array([x.subs(t, time) for time in times_train])).astype("float32")
print(times_train)
print(D_train, "\n")


# create validation data
times_val = (np.random.rand(n) * 0.5).astype("float32")
D_val = (np.array([x.subs(t, time) for time in times_val])).astype("float32")
print(times_val)
print(D_val)

[0.2565484  0.47353992 0.37033448 0.27149448 0.3082405  0.31222937
 0.4422975  0.07040991 0.1248476  0.07147331]
[ 0.62709194  0.03669519 -0.3835772   0.5727451   0.23003878  0.18297355
 -0.22040442 -0.53714454 -0.8530895  -0.55540943] 

[0.1051741  0.1507489  0.01632371 0.35994673 0.04788768 0.07926321
 0.40360487 0.21241547 0.32350183 0.01301449]
[-0.8913805  -0.5795216   0.67723095 -0.31870785 -0.07116849 -0.67622083
 -0.43119153  0.38476053  0.04846714  0.74924797]


In [36]:
simple_model = nn.Sequential(
    # input layer with 1 input
    nn.Linear(1, 10),
    nn.Tanh(),
    # hidden layer
    nn.Linear(10, 1)
)

print(simple_model)
net = simple_model

Sequential(
  (0): Linear(in_features=1, out_features=10, bias=True)
  (1): Tanh()
  (2): Linear(in_features=10, out_features=1, bias=True)
)


In [32]:
optimizer = optim.SGD(net.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [ ]:
# we could have done this ourselves,
# but we should be aware of sklearn and its tools
from sklearn.metrics import accuracy_score

# changing data to tensors
x_train = torch.from_numpy(times_train)
x_val = torch.from_numpy(times_val)
target_train = torch.from_numpy(D_train)
target_val = torch.from_numpy(D_val)

# setting hyperparameters and gettings epoch sizes
batch_size = 10
num_epochs = 100
num_samples_train = len(x_train)
num_batches_train = num_samples_train // batch_size
num_samples_valid = len(target_val)
num_batches_valid = num_samples_valid // batch_size

# setting up lists for handling loss/accuracy
train_acc, train_loss = [], []
valid_acc, valid_loss = [], []
test_acc, test_loss = [], []
cur_loss = 0
losses = []

get_slice = lambda i, size: range(i * size, (i + 1) * size)

for epoch in range(num_epochs):
    # Forward -> Backprob -> Update params
    ## Train
    cur_loss = 0
    net.train()
    for i in range(num_batches_train):
        optimizer.zero_grad()
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        # compute gradients given loss
        target_batch = target_train[slce]
        batch_loss = criterion(output, target_batch)
        batch_loss.backward()
        optimizer.step()
        
        cur_loss += batch_loss   
    losses.append(cur_loss / batch_size)

    net.eval()
    ### Evaluate training
    train_preds, train_targs = [], []
    for i in range(num_batches_train):
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        preds = torch.max(output, 1)[1]
        
        train_targs += list(target_train[slce].numpy())
        train_preds += list(preds.data.numpy())
    
    ### Evaluate validation
    val_preds, val_targs = [], []
    for i in range(num_batches_valid):
        slce = get_slice(i, batch_size)